In [ ]:
import bilby
import h5py
import lalsimulation as lalsim
import lal
import numpy as np
import os

from scipy.interpolate import UnivariateSpline
from scipy.optimize import root_scalar

In [ ]:
outdir = "outdir"
outprefix = "test"
datadir = "/home/selmavangstein/mastersproject/datadir/bilby_output"
filepath = 'chombo/GRChombo_BBSsol02_A17A17q100d17p000_Res40.h5' 
#filepath = 'chombo/GRChombo_BBSsol02_A147A147q100d12p000_Res40.h5'

luminosity_distance = 255. #MPc
mtotal = 71.5 # in solar masses
inclination = 0.78

#for testing reasons
# luminosity_distance = 462.68 #MPc
# mtotal = 74.04 # in solar masses
# inclination = 1.016


beginning_taper_safety = 2
end_taper_safety = 8
pad_factor = 4  # Pad with signal_length on each side

phiRef = 0.0 #reference phase
psi = 1.329 #polarization angle
ra = 2.333 #right ascension 
dec = 0.190 #declination

noise = True
bandpass_filtering = False

## Parameter setup

In [38]:
# Specify the output directory and the name of the simulation.
label = "phenomXP"
bilby.core.utils.setup_logger(outdir=outdir, label=label)
f = h5py.File(filepath, 'r')

In [39]:
# setting up dict to feed parameters into waveforms
params = lal.CreateDict()
lalsim.SimInspiralWaveformParamsInsertNumRelData(params, filepath)

inc = 0 #this needs to be temporarily 0 for tapering calc - will adjust later

# extract masses and convert to different units
m1 = f.attrs['mass1'] #code units
m2 = f.attrs['mass2']

mass_1 = m1 * mtotal / (m1 + m2) #solar masses
mass_2 = m2 * mtotal / (m1 + m2)

# Choose extrinsic parameters

m1SI = mass_1 * lal.MSUN_SI #in kg
m2SI = mass_2 * lal.MSUN_SI

spins = lalsim.SimInspiralNRWaveformGetSpinsFromHDF5File(0., mtotal, filepath)
s1x, s1y, s1z = spins[0], spins[1], spins[2]
s2x, s2y, s2z = spins[3], spins[4], spins[5]

# Set sampling frequency of the data segment that we're going to inject the signal into
# just be aware of aliasing issues if too low
sampling_frequency = 4096.0  # Hz
deltaT = 1.0/sampling_frequency #cadence

# Compute the luminosity distance in SI units
distance = luminosity_distance * lal.PC_SI * 1.0e6


# we need to set the lowest trustable frequency - set as lowest simulated frequency, scaled by the chosen mass. Will change after tapering
f_lower = f.attrs['f_lower_at_1MSUN']/mtotal  # this choice generates the whole NR waveforms from the beginning
fRef = f_lower   #beginning of the waveform
f22_start = f_lower
print(f"f_lower set to {f_lower} Hz")


f_lower set to 20.825324030084587 Hz


In [40]:
f.close()

## Create hp, hc arrays

In [41]:
params = lal.CreateDict()
lalsim.SimInspiralWaveformParamsInsertNumRelData(params, filepath)

approx = lalsim.NR_hdf5

inject_l_modes=[2]
ModeArray = lalsim.SimInspiralCreateModeArray()
for mode in inject_l_modes:
    lalsim.SimInspiralModeArrayActivateAllModesAtL(ModeArray, mode)

lalsim.SimInspiralWaveformParamsInsertModeArray(params, ModeArray)

h_p, h_c = lalsim.SimInspiralChooseTDWaveform(m1SI, m2SI, s1x, s1y, s1z,
                s2x, s2y, s2z, distance, inc, phiRef, np.pi/2., 0.0, 0.0, 
                deltaT, f22_start, fRef, params, approx)

times = np.arange(len(h_p.data.data))*h_p.deltaT

## Tapering

In [42]:
# get stuff for tapering calculation

phase = np.arctan2(h_c.data.data, h_p.data.data)
unwrapped_phase = np.unwrap(phase)
unwrapped_phase_interp = UnivariateSpline(times, unwrapped_phase, k=3, s=0)
omega_interp = unwrapped_phase_interp.derivative() 
frequency = omega_interp(times) / (2.0 * np.pi)

safety = beginning_taper_safety
sol = root_scalar(lambda t1: (1+safety)*(1./t1)-omega_interp(t1)/(2.0*np.pi),
                    x0 = 0.1, 
                    bracket=[0.01, 0.3])
t0, t1 = 0., sol.root
f0, f1 = omega_interp(t0)/(2.0*np.pi), omega_interp(t1)/(2.0*np.pi)
print(f"{t0=:.2e}, {f0=:.2e}, {t1=:.2e}, {f1=:.2e}")

minimum_frequency = f1 #min trustable frequency after tapering


t0=0.00e+00, f0=2.08e+01, t1=1.28e-01, f1=2.34e+01


### Regen hp, hc with right inclination

In [43]:
inc = inclination # setting correct inclination now

h_p, h_c = lalsim.SimInspiralChooseTDWaveform(m1SI, m2SI, s1x, s1y, s1z,
                s2x, s2y, s2z, distance, inc, phiRef, np.pi/2., 0.0, 0.0, 
                deltaT, f22_start, fRef, params, approx)

times = np.arange(len(h_p.data.data))*h_p.deltaT

In [44]:
# just to ensure consistency in time series
from gwpy.timeseries import TimeSeries
new_hp = TimeSeries.from_lal(h_p)
times = new_hp.times.value

#create shifted time array to store hp, hc data with peak at t=0

# find peak index
peak_index = np.argmax(np.abs(h_p.data.data))
peak_time = times[peak_index]
# shift times so that peak is at t=0
times_shift = times - peak_time

#save hp, hc to separate csv files for later plotting
import pandas as pd
hp_df = pd.DataFrame({'times': times_shift, 'strain': h_p.data.data})
hc_df = pd.DataFrame({'times': times_shift, 'strain': h_c.data.data})
hp_df.to_csv('/home/selmavangstein/mastersproject/datadir/bilby_output/hp.csv', index=False)
hc_df.to_csv('/home/selmavangstein/mastersproject/datadir/bilby_output/hc.csv', index=False)

In [45]:
def window(times, t0, t1, direction='on'):
    taper = np.ones_like(times)
    mask = (times >= t0) & (times <= t1)
    x = (times[mask] - t0) / (t1 - t0)
    taper[mask] = 0.5 * (1 - np.cos(np.pi * x))

    if direction == 'off':
        # Flip the taper and set regions outside [t0, t1] appropriately
        taper[mask] = 1 - taper[mask]
        taper[times < t0] = 1.0
        taper[times > t1] = 0.0
    else:  # direction == 'on'
        taper[times < t0] = 0.0
        taper[times > t1] = 1.0
        
    return taper

### Taper beginning

In [46]:
wf = 'cosine' #leftover from previous versions - only cosine implemented

taper = window(times, t0, t1, direction='on')
tapered_hp = h_p.data.data * taper
tapered_hc = h_c.data.data * taper

### Taper end

In [47]:
def find_peaks(hp):
    """
    Find peaks by sign change in derivative
    Returns array of indices (index corresponds to sample where sign change occurs).
    """
    d_hp = np.diff(hp)
    # look for positive -> negative sign change
    sign_change = (d_hp[:-1] > 0) & (d_hp[1:] < 0)
    peaks = np.where(sign_change)[0] + 1
    return peaks

def nth_last_peak(hp, times, n=1):
    peaks = find_peaks(hp)
    nth_idx = peaks[-n]
    nth_time = times[nth_idx]
    nth_value = hp[nth_idx]
    return nth_idx, nth_time, nth_value

n = end_taper_safety
idx_taper = find_peaks(h_p.data.data)[-n]

In [48]:
t_start = times[idx_taper] if idx_taper < len(times) else times[-1]
t_end = times[-1]

if t_start >= t_end:
    raise ValueError(f"Invalid taper interval: t_start ({t_start}) >= t_end ({t_end})")

end_taper = window(times, t_start, t_end, direction='off')
# combine with the existing start taper
taper = taper * end_taper

# update the stored tapers and tapered signals
tapered_hp = tapered_hp * end_taper
tapered_hc = tapered_hc * end_taper

## Padding

In [49]:
signal_length = len(times)
pad_length = int(signal_length * pad_factor)

# Get the time step (assuming uniform sampling)
dt = times[1] - times[0]
pad_time = pad_length * dt

# Create new time array with padding
times_before = times[0] - np.arange(pad_length, 0, -1) * dt
times_after = times[-1] + np.arange(1, pad_length + 1) * dt
# Combine all times
times_padded = np.concatenate([times_before, times, times_after])
# shift to start at t=0
times_padded = times_padded-times_padded[0]

# Pad with zeros on each side
padded_hp = np.pad(tapered_hp, pad_length, mode='constant', constant_values=0)
padded_hc = np.pad(tapered_hc, pad_length, mode='constant', constant_values=0)
padded_taper = np.pad(taper, pad_length, mode='constant', constant_values=0)

print(f"Original signal length: {signal_length} points ({times[0]:.4f} s to {times[-1]:.4f} s)")
print(f"Padded signal length: {len(times_padded)} points ({times_padded[0]:.4f} s to {times_padded[-1]:.4f} s)")
print(f"Padding: {pad_length} points ({pad_length * dt:.4f} s) on each side")
print(f"Signal now occupies {100 * signal_length / len(times_padded):.1f}% of total length")

Original signal length: 2482 points (0.0000 s to 0.6057 s)
Padded signal length: 22338 points (0.0000 s to 5.4534 s)
Padding: 9928 points (2.4238 s) on each side
Signal now occupies 11.1% of total length


### Bandpass filter

In [50]:
# apply band pass filter before injecting into ifos
from scipy.signal import butter, sosfiltfilt
def bandpass_filter(data, lowcut, highcut, fs, order=4):
    sos = butter(order, [lowcut, highcut], btype='band', fs=fs, output='sos')
    y = sosfiltfilt(sos, data)
    return y
lowcut = minimum_frequency  # Hz
highcut = 270  # Hz - by eye
fs = 4096.0  # Sampling frequency in Hz
if bandpass_filtering:
    filtered_hp = bandpass_filter(padded_hp, lowcut, highcut, fs)
    filtered_hc = bandpass_filter(padded_hc, lowcut, highcut, fs)

In [51]:
preprocessed_hp = filtered_hp if (bandpass_filtering) else padded_hp
preprocessed_hc = filtered_hc if (bandpass_filtering) else padded_hc

### Injections

In [52]:
def create_nr_injection(times_array, hp_array, hc_array):
    """
    Factory function that creates a custom NR injection function
    with specific time series data.
    """
    def nr_injection(time):
            
        hp = np.interp(time, times_array, hp_array)
        hc = np.interp(time, times_array, hc_array)
        return {"plus": hp, "cross": hc}
    
    return nr_injection

In [53]:
# propagate and inject each tapered signal
# create dict to store the data

# save original hp, hc for reference
original_hp = np.copy(h_p.data.data)
original_hc = np.copy(h_c.data.data)

#identify merger peak and corresponding phase
amplitude = np.sqrt(preprocessed_hp**2 + preprocessed_hc**2)
peak_id = np.argmax(amplitude)
peak = times_padded[peak_id]

hplus_peak = preprocessed_hp[peak_id]
hcross_peak = preprocessed_hc[peak_id]
phase_merger = np.arctan2(-hcross_peak,hplus_peak) + np.pi


#duration = times_padded[-1]
start_time = times_padded[0]
signal_start_time = times_padded[pad_length] #where signal turns on 
duration = len(times_padded) / sampling_frequency

np.random.seed(88170235)
injection_parameters = dict(
    mass_1=mass_1,
    mass_2=mass_2,
    a_1=0.0,
    a_2=0.0,
    tilt_1=0.0,
    tilt_2=0.0,
    phi_12=0.0,
    phi_jl=0.0,
    luminosity_distance=luminosity_distance,
    theta_jn=inc,
    psi=psi,
    phase=phase_merger,
    geocent_time=0, #will adjust for time shift later
    ra=ra,
    dec=dec
)

# bibly assumes a signal centered at 0 and then applies a geocentric time shift, so we must shift our signal accordingly
# roll the arrays so the peak is at index 0
#hp_rolled = np.roll(preprocessed_hp, -peak_id)
#hc_rolled = np.roll(preprocessed_hc, -peak_id)
# edit: changed convention here - no rolling, just use as is
hp_rolled = preprocessed_hp
hc_rolled = preprocessed_hc

nr_injection_wf = create_nr_injection(times_padded, hp_rolled, hc_rolled)

# propagate to interferometers
waveform = bilby.gw.waveform_generator.WaveformGenerator(duration=duration, sampling_frequency=sampling_frequency,
        time_domain_source_model=nr_injection_wf, start_time=0);

# get time and frequency domain strains and arrays
time_domain = waveform.time_domain_strain(parameters=injection_parameters);
time_array = waveform.time_array

fr_domain = waveform.frequency_domain_strain(parameters=injection_parameters);
fr_array = waveform.frequency_array

15:25 bilby INFO    : Waveform generator instantiated: WaveformGenerator(duration=5.45361328125, sampling_frequency=4096.0, start_time=0, frequency_domain_source_model=None, time_domain_source_model=__main__.nr_injection, parameter_conversion=bilby.gw.conversion.convert_to_lal_binary_black_hole_parameters, waveform_arguments={})


In [54]:
# create ifos and inject signal and noise
ifos = bilby.gw.detector.InterferometerList(['H1', 'L1']);
if noise:
    ifos.set_strain_data_from_power_spectral_densities(
        sampling_frequency=sampling_frequency, duration=duration,
        start_time=start_time);
else:
    ifos.set_strain_data_from_zero_noise(
        sampling_frequency=sampling_frequency, duration=duration,
        start_time=start_time);

ifos.inject_signal(waveform_generator=waveform,
                parameters=injection_parameters, raise_error=False);

15:25 bilby INFO    : Injected signal in H1:
15:25 bilby INFO    :   optimal SNR = 58.09
15:25 bilby INFO    :   matched filter SNR = 58.73-0.51j
15:25 bilby INFO    :   mass_1 = 35.75
15:25 bilby INFO    :   mass_2 = 35.75
15:25 bilby INFO    :   a_1 = 0.0
15:25 bilby INFO    :   a_2 = 0.0
15:25 bilby INFO    :   tilt_1 = 0.0
15:25 bilby INFO    :   tilt_2 = 0.0
15:25 bilby INFO    :   phi_12 = 0.0
15:25 bilby INFO    :   phi_jl = 0.0
15:25 bilby INFO    :   luminosity_distance = 255.0
15:25 bilby INFO    :   theta_jn = 0.78
15:25 bilby INFO    :   psi = 1.329
15:25 bilby INFO    :   phase = 4.052962732862897
15:25 bilby INFO    :   geocent_time = 0
15:25 bilby INFO    :   ra = 2.333
15:25 bilby INFO    :   dec = 0.19
15:25 bilby INFO    : Injected signal in L1:
15:25 bilby INFO    :   optimal SNR = 56.13
15:25 bilby INFO    :   matched filter SNR = 57.24+0.08j
15:25 bilby INFO    :   mass_1 = 35.75
15:25 bilby INFO    :   mass_2 = 35.75
15:25 bilby INFO    :   a_1 = 0.0
15:25 bilby I

## Time Shifting and Data Export

Now we shift the time arrays so that:
1. The reference geocenter time is set to match GW150914 paper values (1420878141.235932 GPS)
2. Detector arrival times are calculated accounting for light travel time from geocenter
3. Peak amplitudes in each detector align with their respective arrival times
4. All saved data uses GPS time coordinates consistent with the ringdown analysis script

In [ ]:
# === TIME SHIFTING AND DATA SAVING ===
# Set reference geocenter time (using the paper's value for consistency)
reference_geocent_time = 1420878141.235932  # GPS time from GW150914 paper Table II

# Calculate arrival times at each detector
h1_delay = ifos[0].time_delay_from_geocenter(ra, dec, reference_geocent_time)
l1_delay = ifos[1].time_delay_from_geocenter(ra, dec, reference_geocent_time)

h1_arrival_time = reference_geocent_time + h1_delay
l1_arrival_time = reference_geocent_time + l1_delay

# Find peak in each detector's strain
h1_strain = ifos[0].time_domain_strain
l1_strain = ifos[1].time_domain_strain
h1_time_array = ifos[0].time_array
l1_time_array = ifos[1].time_array

h1_amplitude = np.abs(h1_strain)
l1_amplitude = np.abs(l1_strain)
h1_peak_idx = np.argmax(h1_amplitude)
l1_peak_idx = np.argmax(l1_amplitude)

h1_peak_time_relative = h1_time_array[h1_peak_idx]
l1_peak_time_relative = l1_time_array[l1_peak_idx]

# Calculate shifts needed to align peaks with arrival times
# The shifted time array should have: peak_time_shifted = arrival_time
# So: time_original + shift = arrival_time at peak index
# Therefore: shift = arrival_time - peak_time_original
h1_time_shift = h1_arrival_time - h1_peak_time_relative
l1_time_shift = l1_arrival_time - l1_peak_time_relative

# Create shifted time arrays - arrays created from scratch to ensure uniform spacing
n_samples = len(h1_strain)
dt = 1.0 / sampling_frequency  # Exact time step

# Create start and end times for H1
h1_start_time = h1_arrival_time - h1_peak_idx * dt
h1_end_time = h1_start_time + (n_samples - 1) * dt
# linspace ensures perfect uniform spacing independent of accumulation errors
h1_time_shifted = np.linspace(h1_start_time, h1_end_time, n_samples)

# Create start and end times for L1
l1_start_time = l1_arrival_time - l1_peak_idx * dt
l1_end_time = l1_start_time + (n_samples - 1) * dt
# linspace for L1 as well
l1_time_shifted = np.linspace(l1_start_time, l1_end_time, n_samples)

# Verify the peak is at the correct time
print(f"\nPeak locations after shifting:")
print(f"H1 peak at: {h1_time_shifted[h1_peak_idx]:.6f} GPS (should be {h1_arrival_time:.6f})")
print(f"L1 peak at: {l1_time_shifted[l1_peak_idx]:.6f} GPS (should be {l1_arrival_time:.6f})")
print(f"Goal sampling frequency: {sampling_frequency} Hz")
print(f"Sampling freq from H1 times: {1.0 / (h1_time_shifted[1] - h1_time_shifted[0]):.17g} Hz")

# Update injection_parameters with the reference geocenter time
injection_parameters['geocent_time'] = reference_geocent_time



=== TIME CONFIGURATION ===
Reference geocenter time: 1420878141.235932 GPS
H1 arrival time: 1420878141.219014 GPS
L1 arrival time: 1420878141.216519 GPS
H1 delay from geocenter: -16.918 ms
L1 delay from geocenter: -19.413 ms

Peak locations in original time array:
H1 peak at: 2.965332 s (relative)
L1 peak at: 2.965820 s (relative)

Time shifts applied:
H1 shift: 1420878138.253682 s
L1 shift: 1420878138.250699 s

Peak locations after shifting:
H1 peak at: 1420878141.219014 GPS (should be 1420878141.219014)
L1 peak at: 1420878141.216519 GPS (should be 1420878141.216519)
Goal sampling frequency: 4096.0 Hz
Sampling freq from H1 times: 4096 Hz


In [57]:
# Save data with shifted time arrays
import pandas as pd
import os

os.makedirs(datadir, exist_ok=True)

# For ringdown analysis - save with H1 time (since ringdown uses H1 as reference)
# Save time array with full precision to preserve uniformity
df_h1_reference = pd.DataFrame({
    'time': h1_time_shifted,
    'H1': h1_strain,
    'L1': l1_strain
})


injection_csv = os.path.join(datadir, f'{outprefix}_injection_data.csv')
df_h1_reference.to_csv(injection_csv, index=False, float_format='%.17g')
print(f"\nSaved shifted data to '{injection_csv}'")


Saved shifted data to '/home/selmavangstein/mastersproject/datadir/bilby_output/test_injection_data.csv'


In [59]:
print("\n=== METADATA FOR RINGDOWN FIT ===")
print(f"mass:              {mtotal} Msun")
print(f"luminosity dist:   {injection_parameters['luminosity_distance']} Mpc")
print(f"ra:                {injection_parameters['ra']}")
print(f"dec:               {injection_parameters['dec']}")
print(f"psi:               {injection_parameters['psi']}")
print(f"duration:          {duration}")
print(f"H1 SNR:            {ifo.meta_data['optimal_SNR']:.2f}")
print(f"L1 SNR:            {ifos[1].meta_data['optimal_SNR']:.2f}")
print(f"geocent time:      {reference_geocent_time}")
print(f"h1 arrival time:   {h1_arrival_time}")
print(f"h1 peak time:      {h1_time_shifted[h1_peak_idx]}")
print("=================================")


=== METADATA FOR RINGDOWN FIT ===
mass:              71.5 Msun
luminosity dist:   255.0 Mpc
ra:                2.333
dec:               0.19
psi:               1.329
duration:          5.45361328125
H1 SNR:            58.09
L1 SNR:            56.13
geocent time:      1420878141.235932
h1 arrival time:   1420878141.2190142
h1 peak time:      1420878141.2190142


## Create dictionary of metadata used for later analysis

In [60]:
meta_dict = {
    "total_mass": mtotal,
    "mass_ratio": mass_1/mass_2,
    "luminosity_distance": injection_parameters['luminosity_distance'],
    "ra": injection_parameters['ra'],
    "dec": injection_parameters['dec'],
    "psi": injection_parameters['psi'],
    "duration": duration,
    "f_min": minimum_frequency, #lowest trustable frequency after tapering
    "f_max": highcut, # by eye highest trustable frequency
    "f_ref": fRef, # same as f_lower, used as fRef in injections
    "f22_start": f_lower, #from lowest simulated frequency (from raw waveform before tapering)
    "inclination": inc,
    "phase": injection_parameters['phase'],
    "geocenter_time": reference_geocent_time,
    "h1_arrival_time": h1_arrival_time,
    "l1_arrival_time": l1_arrival_time,
    "h1_peak_time": h1_time_shifted[h1_peak_idx],
    "l1_peak_time": l1_time_shifted[l1_peak_idx],
    "sampling_frequency": sampling_frequency,
    "start_time": h1_time_shifted[0],
    "pad_time": pad_time,
}

df = pd.DataFrame([meta_dict])
metadata_csv = os.path.join(datadir, f'{outprefix}_injection_metadata.csv')
df.to_csv(metadata_csv, index=False)
print(f"Saved metadata to '{metadata_csv}'")
print(f"Note: f22_start={f_lower:.2f} Hz (raw waveform), f_min={minimum_frequency:.2f} Hz (after tapering)")

Saved metadata to '/home/selmavangstein/mastersproject/datadir/bilby_output/test_injection_metadata.csv'
Note: f22_start=20.83 Hz (raw waveform), f_min=23.38 Hz (after tapering)


## Generate PSD files for TDInf

In [ ]:
h1_psd = ifos[0].power_spectral_density_array
h1_freq = ifos[0].frequency_array

l1_psd = ifos[1].power_spectral_density_array
l1_freq = ifos[1].frequency_array

#replace inf with 0.25 in psd
h1_psd[np.isinf(h1_psd)] = 0.25
l1_psd[np.isinf(l1_psd)] = 0.25

# create array of pairs (freq, psd) for each ifo
h1_psd_array = np.column_stack((h1_freq, h1_psd))
l1_psd_array = np.column_stack((l1_freq, l1_psd))


#save .dat files to datadir (for consistency, PSDs are by ifo so never changed)
import os

h1_psd = os.path.join(datadir, f'H1_PSD.dat')
l1_psd = os.path.join(datadir, f'L1_PSD.dat')


# save each array to .dat file
np.savetxt(h1_psd, h1_psd_array)
np.savetxt(l1_psd, l1_psd_array)

## Save injection data on right format for TDInf

In [ ]:
# === CREATE SPLIT INJECTION FILES (CSV and HDF5) ===
import os

# Create split CSV files
h1_csv = os.path.join(datadir, f'{outprefix}_H1_injection_data.csv')
l1_csv = os.path.join(datadir, f'{outprefix}_L1_injection_data.csv')

H1_df = pd.DataFrame({'times': h1_time_shifted, 'strain': h1_strain})
L1_df = pd.DataFrame({'times': l1_time_shifted, 'strain': l1_strain})

H1_df.to_csv(h1_csv, index=False, float_format='%.17g')
L1_df.to_csv(l1_csv, index=False, float_format='%.17g')
print(f"Saved H1 injection data to '{h1_csv}'")
print(f"Saved L1 injection data to '{l1_csv}'")

# Create split HDF5 files
h1_h5 = os.path.join(datadir, f'{outprefix}_H1_injection_data.h5')
l1_h5 = os.path.join(datadir, f'{outprefix}_L1_injection_data.h5')

with h5py.File(h1_h5, 'w') as f:
    f.create_dataset('times', data=h1_time_shifted)
    f.create_dataset('strain', data=h1_strain)
with h5py.File(l1_h5, 'w') as f:
    f.create_dataset('times', data=l1_time_shifted)
    f.create_dataset('strain', data=l1_strain)

print(f"Saved H1 injection data to '{h1_h5}'")
print(f"Saved L1 injection data to '{l1_h5}'")

Saved H1 injection data to '/home/selmavangstein/mastersproject/datadir/bilby_output/test_H1_injection_data.csv'
Saved L1 injection data to '/home/selmavangstein/mastersproject/datadir/bilby_output/test_L1_injection_data.csv'
Saved H1 injection data to '/home/selmavangstein/mastersproject/datadir/bilby_output/test_H1_injection_data.h5'
Saved L1 injection data to '/home/selmavangstein/mastersproject/datadir/bilby_output/test_L1_injection_data.h5'

Verification:
H1 CSV: 922.5 KB
L1 CSV: 922.5 KB
H1 HDF5: 351.0 KB
L1 HDF5: 351.0 KB
